In [1]:
import pyspark
from pyspark.sql import SparkSession

In [2]:
spark = SparkSession.builder.config('spark.driver.host','localhost').appName('03').getOrCreate()

In [3]:
# fifa19.csv로부터 Name, Age, Nationality, Club, Overall만 추출해 fifa에 저장
path = 'C:\\datas\\3rd_week_data\\fifa19.csv'
fifa = spark.read.csv(path,header=True,inferSchema=True).select('Name','Age','Nationality','Club','Overall')
fifa.printSchema()
fifa.show(5)

root
 |-- Name: string (nullable = true)
 |-- Age: integer (nullable = true)
 |-- Nationality: string (nullable = true)
 |-- Club: string (nullable = true)
 |-- Overall: integer (nullable = true)

+-----------------+---+-----------+-------------------+-------+
|             Name|Age|Nationality|               Club|Overall|
+-----------------+---+-----------+-------------------+-------+
|         L. Messi| 31|  Argentina|       FC Barcelona|     94|
|Cristiano Ronaldo| 33|   Portugal|           Juventus|     94|
|        Neymar Jr| 26|     Brazil|Paris Saint-Germain|     92|
|           De Gea| 27|      Spain|  Manchester United|     91|
|     K. De Bruyne| 27|    Belgium|    Manchester City|     91|
+-----------------+---+-----------+-------------------+-------+
only showing top 5 rows



## `Column`명 변경하기

- `DataFrame.withColumnRenamed('old','new')`
- **주의**: `SQL` 명령을 사용할 때, **Column명에 공백이 없어야 함**

In [4]:
fifa = fifa.withColumnRenamed('Name','Full_Name').withColumnRenamed('Club','Club_Name')
fifa.printSchema()

root
 |-- Full_Name: string (nullable = true)
 |-- Age: integer (nullable = true)
 |-- Nationality: string (nullable = true)
 |-- Club_Name: string (nullable = true)
 |-- Overall: integer (nullable = true)



## `SparkSession.sql`

- `sql` 작업을 하기 위해서는 데이터베이스의 가상 테이블은 `view`를 생성해야 함
- `DataFrame.createOrReplaceTempView('view_name')`
> - Temporary: 메모리 상에 잠시 머물다가, SparkSession 종료 후에 삭제됨
> - DataFrame에 SQL용 별명을 붙여서, 데이터베이스의 table처럼 취급하게 해주는 가상의 창구

In [5]:
fifa.createOrReplaceTempView('fifa_view')

- `SparkSession.sql('view를 이용한 SQL명령을 입력')`: 결과는 `DataFrame`으로 반환

> - `SELECT`와`WHERE`: `WHERE`=조건으로 값 조회

In [6]:
# Age가 35보다 큰 행의 전체열을 추출
spark.sql('SELECT * FROM fifa_view WHERE Age>35').show(5)

+--------------+---+-----------+--------------------+-------+
|     Full_Name|Age|Nationality|           Club_Name|Overall|
+--------------+---+-----------+--------------------+-------+
|     G. Buffon| 40|      Italy| Paris Saint-Germain|     88|
|Z. Ibrahimović| 36|     Sweden|           LA Galaxy|     85|
|   A. Barzagli| 37|      Italy|            Juventus|     84|
|   David Villa| 36|      Spain|    New York City FC|     82|
|        Aduriz| 37|      Spain|Athletic Club de ...|     82|
+--------------+---+-----------+--------------------+-------+
only showing top 5 rows



In [7]:
# 2개 조건 조합
# 35세보다 많고(Age>35), 국적이 Spain(Nationality=='Spain')인 선수의 전체 열 추출
spark.sql("SELECT * FROM fifa_view WHERE Age>35 AND Nationality == 'Spain'").show(5)

+-----------+---+-----------+--------------------+-------+
|  Full_Name|Age|Nationality|           Club_Name|Overall|
+-----------+---+-----------+--------------------+-------+
|David Villa| 36|      Spain|    New York City FC|     82|
|     Aduriz| 37|      Spain|Athletic Club de ...|     82|
|   Casillas| 37|      Spain|            FC Porto|     82|
|    Joaquín| 36|      Spain|          Real Betis|     81|
|Diego López| 36|      Spain|        RCD Espanyol|     80|
+-----------+---+-----------+--------------------+-------+
only showing top 5 rows



> - `GROUP BY`: 특정한 열의 값을 기준으로 grouping한 후, group별 특정 값을 집계함(mean, count, sum, ...)

In [8]:
# 각 Club별 선수들의 Overall 평균값을 DataFrame으로 보이시오
spark.sql("SELECT Club_Name, MEAN(Overall) FROM fifa_view GROUP BY Club_Name").show(10)

+--------------------+------------------+
|           Club_Name|     mean(Overall)|
+--------------------+------------------+
|             Palermo| 67.96666666666667|
|          Göztepe SK| 68.42857142857143|
|CD Everton de Viñ...|63.666666666666664|
|     Shonan Bellmare|              60.6|
|          Sagan Tosu| 64.23333333333333|
|  1. FC Union Berlin| 68.32142857142857|
|               Carpi|63.666666666666664|
|           Puebla FC| 65.23333333333333|
|  Argentinos Juniors| 64.64285714285714|
|     SC Paderborn 07|63.733333333333334|
+--------------------+------------------+
only showing top 10 rows



In [9]:
# AS를 통해 column명 변경
spark.sql("SELECT Club_Name, MEAN(Overall) AS Club_Overall FROM fifa_view GROUP BY Club_Name").show(10)

+--------------------+------------------+
|           Club_Name|      Club_Overall|
+--------------------+------------------+
|             Palermo| 67.96666666666667|
|          Göztepe SK| 68.42857142857143|
|CD Everton de Viñ...|63.666666666666664|
|     Shonan Bellmare|              60.6|
|          Sagan Tosu| 64.23333333333333|
|  1. FC Union Berlin| 68.32142857142857|
|               Carpi|63.666666666666664|
|           Puebla FC| 65.23333333333333|
|  Argentinos Juniors| 64.64285714285714|
|     SC Paderborn 07|63.733333333333334|
+--------------------+------------------+
only showing top 10 rows



## `ml.feature.SQLTransformer`

- `SQLTransformer`는 view를 생성하지 않고, SQL 명령을 사용할 수 있음 (SQL 구문 자체를 파이프라인 단계로 만든느 것)
- 객체 생성 `SQLTransformer(statement)`: 하나의 파라미터(SQL 쿼리 문자열)만 사용
- 객체 생성 후, DataFrame에 적용: `SQLTransformer.transform(DataFrame)`

In [10]:
from pyspark.ml.feature import SQLTransformer

In [11]:
# SQL 명령 문자열을 통해 SQLTransformer를 생성
# __THIS__: 입력받을 DataFrame을 의미함
# ex> Club_Name 열을 추출
sql_tf = SQLTransformer(statement="SELECT Club_Name FROM __THIS__")
print(type(sql_tf))

<class 'pyspark.ml.feature.SQLTransformer'>


In [12]:
# SQLTransformer.transform(dataframe)을 통해 적용
sql_tf.transform(fifa).show(10)

+-------------------+
|          Club_Name|
+-------------------+
|       FC Barcelona|
|           Juventus|
|Paris Saint-Germain|
|  Manchester United|
|    Manchester City|
|            Chelsea|
|        Real Madrid|
|       FC Barcelona|
|        Real Madrid|
|    Atlético Madrid|
+-------------------+
only showing top 10 rows



In [13]:
# Age가 40보다 큰 선수들의 Full_Name, Age, Overall 추출
sql_tf2 = SQLTransformer(statement="SELECT Full_Name, Age, Overall FROM __THIS__ WHERE Age>40")
sql_tf2.transform(fifa).show(10)

+-------------+---+-------+
|    Full_Name|Age|Overall|
+-------------+---+-------+
|    J. Villar| 41|     77|
|     B. Nivet| 41|     71|
|     O. Pérez| 45|     71|
|     C. Muñoz| 41|     68|
|  S. Narazaki| 42|     65|
| H. Sulaimani| 41|     63|
|     M. Tyler| 41|     59|
|    T. Warner| 44|     53|
|K. Pilkington| 44|     48|
+-------------+---+-------+



## `sql.functions.expr`

- SQL 문법을 통해 새로운 **Column**을 반환함
- `SQLTransformer`: 주로 ML용, DataFrame 전체를 변환
- `sql.functions.expr`: 일반 데이터 처리용, 개별 column을 계산 (`select, withColumn, filter` 등 내부에서 사용)

- `DataFrame.withColumn(column명, column값)`: `DataFrame`에 **Column값**을 갖는 **column명** column을 생성함
> - 이 때, **column값**에 `expr`을 적용하여 값을 할당할 수 있음

In [17]:
from pyspark.sql.functions import expr

In [18]:
# Overall 에서 Age를 뺀 결과를 Value라는 열로 생성
fifa.withColumn("Value",expr("Overall - Age")).show(10)

+-----------------+---+-----------+-------------------+-------+-----+
|        Full_Name|Age|Nationality|          Club_Name|Overall|Value|
+-----------------+---+-----------+-------------------+-------+-----+
|         L. Messi| 31|  Argentina|       FC Barcelona|     94|   63|
|Cristiano Ronaldo| 33|   Portugal|           Juventus|     94|   61|
|        Neymar Jr| 26|     Brazil|Paris Saint-Germain|     92|   66|
|           De Gea| 27|      Spain|  Manchester United|     91|   64|
|     K. De Bruyne| 27|    Belgium|    Manchester City|     91|   64|
|        E. Hazard| 27|    Belgium|            Chelsea|     91|   64|
|        L. Modrić| 32|    Croatia|        Real Madrid|     91|   59|
|        L. Suárez| 31|    Uruguay|       FC Barcelona|     91|   60|
|     Sergio Ramos| 32|      Spain|        Real Madrid|     91|   59|
|         J. Oblak| 25|   Slovenia|    Atlético Madrid|     90|   65|
+-----------------+---+-----------+-------------------+-------+-----+
only showing top 10 

In [21]:
# Full_Name과 Overall을 결합한 Name_Overall 새로운 열 생성
fifa.withColumn('Name_Overall',expr("CONCAT(Full_Name,'+',Overall)")).show(10)

+-----------------+---+-----------+-------------------+-------+--------------------+
|        Full_Name|Age|Nationality|          Club_Name|Overall|        Name_Overall|
+-----------------+---+-----------+-------------------+-------+--------------------+
|         L. Messi| 31|  Argentina|       FC Barcelona|     94|         L. Messi+94|
|Cristiano Ronaldo| 33|   Portugal|           Juventus|     94|Cristiano Ronaldo+94|
|        Neymar Jr| 26|     Brazil|Paris Saint-Germain|     92|        Neymar Jr+92|
|           De Gea| 27|      Spain|  Manchester United|     91|           De Gea+91|
|     K. De Bruyne| 27|    Belgium|    Manchester City|     91|     K. De Bruyne+91|
|        E. Hazard| 27|    Belgium|            Chelsea|     91|        E. Hazard+91|
|        L. Modrić| 32|    Croatia|        Real Madrid|     91|        L. Modrić+91|
|        L. Suárez| 31|    Uruguay|       FC Barcelona|     91|        L. Suárez+91|
|     Sergio Ramos| 32|      Spain|        Real Madrid|     91|  

- `expr`과 `select`의 결합
- `select` + `expr`은 `where` 및 `filter`와 유사한 기능 + 열 생성
- 소수점 3자리까지 보려면 `round(column,3)`과 같은 형식으로 작성

In [20]:
# Age가 30보다 작고, Overall이 80보다 큰 선수는 True, 나머지 선수는 False가 되는 High_Value열 생성
fifa.select("*",expr("Age<30 AND Overall>80 AS High_Value")).show(5)
fifa.selectExpr("*","Age<30 AND Overall>80 AS High_Value").show(5)

+-----------------+---+-----------+-------------------+-------+----------+
|        Full_Name|Age|Nationality|          Club_Name|Overall|High_Value|
+-----------------+---+-----------+-------------------+-------+----------+
|         L. Messi| 31|  Argentina|       FC Barcelona|     94|     false|
|Cristiano Ronaldo| 33|   Portugal|           Juventus|     94|     false|
|        Neymar Jr| 26|     Brazil|Paris Saint-Germain|     92|      true|
|           De Gea| 27|      Spain|  Manchester United|     91|      true|
|     K. De Bruyne| 27|    Belgium|    Manchester City|     91|      true|
+-----------------+---+-----------+-------------------+-------+----------+
only showing top 5 rows



#### 과제
- `SQLTransformer`를 사용하여, **Nationality**에 대해 **선수의 수**와 **Overall 평균**을 보이시오
- **선수의 수**에 해당하는 열의 이름은 **Players**, **Overall 평균**에 해당하는 열의 이름은 **Overall**로 하시오

In [23]:
ex_1 = SQLTransformer(statement="SELECT Nationality,COUNT(Full_Name) AS Players,MEAN(Overall) AS Overall FROM __THIS__ GROUP BY Nationality")
ex_1.transform(fifa).show(5)

+-----------+-------+------------------+
|Nationality|Players|           Overall|
+-----------+-------+------------------+
|       Chad|      2|              70.0|
|     Russia|     79|  70.0632911392405|
|   Paraguay|     85| 69.61176470588235|
|    Senegal|    130| 68.61538461538461|
|     Sweden|    397|63.622166246851386|
+-----------+-------+------------------+
only showing top 5 rows



#### 과제
- `select`와 `expr`을 사용하여 **Overall** 값을 10점 만점으로 변경하고, **10_Overall**이라는 이름으로 생성하시오

In [24]:
fifa.select("*",expr("Overall/10 AS 10_Overall")).show(5)

+-----------------+---+-----------+-------------------+-------+----------+
|        Full_Name|Age|Nationality|          Club_Name|Overall|10_Overall|
+-----------------+---+-----------+-------------------+-------+----------+
|         L. Messi| 31|  Argentina|       FC Barcelona|     94|       9.4|
|Cristiano Ronaldo| 33|   Portugal|           Juventus|     94|       9.4|
|        Neymar Jr| 26|     Brazil|Paris Saint-Germain|     92|       9.2|
|           De Gea| 27|      Spain|  Manchester United|     91|       9.1|
|     K. De Bruyne| 27|    Belgium|    Manchester City|     91|       9.1|
+-----------------+---+-----------+-------------------+-------+----------+
only showing top 5 rows



#### **과제**
- `Age`가 38이상인 선수의 `Nationality, Name, Overall, Age`를 `Age`를 기준으로 내림차순 정렬 후, 상위 10개 행을 보이시오.

#### **과제**
- `Age`가 20세 미만인 선수 중, `Potential`이 가장 큰 선수의 `Nationality`를 찾으시오.
- `DataFrame`에 `Potential, Age, Nationality`가 포함되도록 다시 추출하시오.